In [1]:
# 주요 라이브러리 설치 및 실행 환경 확인
import sys            # Python 실행 환경 확인을 위한 라이브러리 불러오기
import torch          # PyTorch 라이브러리 
import transformers   # Transformers 라이브러리 
import pandas         # Pandas 라이브러리 

print(sys.executable)       # 현재 사용 중인 Python 실행 파일 경로 확인
print(torch.__version__)         # PyTorch 버전 확인
print(transformers.__version__)  # Transformers 버전 확인
print(pandas.__version__)        # Pandas 버전 확인
print(torch.cuda.is_available()) # CUDA GPU 사용 가능 여부 확인


c:\python-llm-advanced\.venv\Scripts\python.exe
2.14.1+cpu
4.57.6
2.3.3
False


In [2]:
# ==================================================
# 실습 1: Ollama 서버 연결 확인
# ==================================================

# Python에서 HTTP 요청
import requests

# Ollama 서버 주소
# localhost: 현재 사용 중인 컴퓨터
# 11434: Ollama의 기본 포트 번호
url = "http://localhost:11434/api/tags"

# requests.get(): HTTP GET 요청
# timeout=10: 최대 10초 동안 응답 대기
response = requests.get(url, timeout=10)

# HTTP 오류 발생 시 예외 처리
response.raise_for_status()

# 서버가 반환한 JSON 데이터를 딕셔너리로 변환
data = response.json()

# for 반복문으로 모델 목록 출력
# .get(): 딕셔너리에서 키에 해당하는 값을 가져오는 메서드
for model in data.get("models", []):
    print(model["name"])

gemma3:1b
qwen3:0.6b


In [17]:
# ==================================================
# 실습 2: Python에서 Qwen3 LLM 호출
# ==================================================

import requests
# 시간 측정 및 대기 기능을 위한 라이브러리
import time

# [LLM 개념]
# MODEL: 사용할 LLM 모델 이름
# URL: Ollama의 대화형 API 주소
MODEL = "qwen3:0.6b"
URL = "http://localhost:11434/api/chat"

# 딕셔너리(dict)는 key:value 구조로 데이터를 저장, messages는 대화 내용을 리스트로 관리
# Ollama API에 전달할 요청 데이터 생성
payload = {
    "model": MODEL,
    "messages": [{
            "role": "user",
            "content": "생성형 AI와 기존 인공지능의 차이를 정확히 세 문장으로 설명해줘."
        }],
    "stream": False,
    "think": False
}

# [실행 시간 측정]
# perf_counter(): 경과 시간을 정밀하게 측정하는 함수

start_time = time.perf_counter()

# [HTTP POST]
# json=payload: 딕셔너리를 JSON으로 변환하여 서버에 전송

response = requests.post(
    URL,
    json=payload,
    timeout=180
)

# HTTP 오류 확인
response.raise_for_status()

# JSON 응답을 딕셔너리로 변환
result = response.json()

# 실행 시간 계산
elapsed = time.perf_counter() - start_time

# [파이썬 문법]
# 중첩 딕셔너리에서 응답 내용 추출

answer = result["message"]["content"]

# 결과 출력
print("모델:", MODEL)
print("응답:", answer)
print(f"실행 시간: {elapsed:.2f}초")

모델: qwen3:0.6b
응답: 생성형 AI는 기존 인공지능보다 더 구체적인 정보를 제공해 줍니다. 기존 AI는 일반적인 데이터를 기반으로 기능하는 것이며, 생성형 AI는 새로운 데이터를 수집하고 분석해 줍니다. 이는 AI 기능의 방식이 달라지게 됩니다.
실행 시간: 4.24초


In [18]:
print(result)

{'model': 'qwen3:0.6b', 'created_at': '2026-11-01T11:36:56.05911Z', 'message': {'role': 'assistant', 'content': '생성형 AI는 기존 인공지능보다 더 구체적인 정보를 제공해 줍니다. 기존 AI는 일반적인 데이터를 기반으로 기능하는 것이며, 생성형 AI는 새로운 데이터를 수집하고 분석해 줍니다. 이는 AI 기능의 방식이 달라지게 됩니다.'}, 'done': True, 'done_reason': 'stop', 'total_duration': 2210947700, 'load_duration': 1181147300, 'prompt_eval_count': 42, 'prompt_eval_cached_count': 0, 'prompt_eval_duration': 97990000, 'eval_count': 75, 'eval_duration': 923978000}


In [9]:
# 질문을 변경하여 동일한 LLM에 요청

questions = [
    "파이썬의 리스트와 튜플의 차이는?",
    "파이썬에서 클래스와 객체의 차이는?",
    "AI 에이전트와 일반 챗봇의 차이는?"
]

# [파이썬 문법]
# for 반복문: 리스트의 질문을 순서대로 처리
# enumerate(): 반복 순서와 값을 함께 반환

for i, question in enumerate(questions, start=1):

    payload["messages"] = [
        {
            "role": "user",
            "content": question
        }
    ]

    response = requests.post(
        URL,
        json=payload,
        timeout=180
    )

    response.raise_for_status()

    result = response.json()

    print(f"\n질문 {i}: {question}")
    print("답변:", result["message"]["content"])
    print("-" * 50)


질문 1: 파이썬의 리스트와 튜플의 차이는?
답변: 파이썬의 `list`와 `tuple`는 **데이터 구조**에서 **차이**가 있습니다. 아래에 그 차이를 정리해 드릴게요:

---

### 1. **데이터 구조**
- `list`: `list`은 **리스트**로 구조를 보여주는 데이터 구조입니다.
- `tuple`: `tuple`은 **튜플**로 구조를 보여주는 데이터 구조입니다.

---

### 2. **데이터의 주소 (Address)**
- `list`는 `tuple`와 비교하면 `list`의 주소가 **더 크다**.  
  예:  
  ```python
  list = [1, 2, 3]
  tuple = (1, 2, 3)
  print(id(list), id(tuple))  # (12, 123)
  ```

---

### 3. **변수의 주소 (Address)**
- `tuple`는 `list`와 **same** 주소를 가집니다.  
  따라서 `tuple`에서 `list`의 내용을 참조하려면 `tuple`의 주소를 참조해야 합니다.

---

### 4. **데이터의 손실 (Loss)**
- `tuple`는 `list`의 내용을 참조할 수 없기 때문에 **데이터 손실**이 발생합니다.

---

### 5. **복잡성**
- `tuple`는 **복잡한 데이터**를 저장할 수 있지만, `list`는 복잡한 데이터를 저장할 수 있습니다.

---

### 6. **데이터의 유사성**
- `tuple`은 `list`과 **유사**한 구조를 가집니다.
- 그러나 `tuple`는 `list`의 내용을 **추출할 수 없기 때문에** `tuple`에서 `list`의 내용을 참조할 수 없습니다.

---

### 7. **데이터의 저장 공간**
- `list`는 `tuple`와 비교하면 **더 작은 공간**을 사용합니다.

---

### ✅ 결론:
| 구분 | list | tuple |
|------|-----|------|
| 데이터 구조 | 리스트 | 

In [8]:
print( response.raise_for_status())
print(response.json())

None
{'model': 'qwen3:0.6b', 'created_at': '2026-11-01T07:18:34.5870912Z', 'message': {'role': 'assistant', 'content': 'AI 에이전트와 일반 챗봇의 차이는 다음과 같습니다:\n\n### 1. **개인 맞춤형 서비스**\n- AI 에이전트는 특정 목표, 유저, 또는 분야에 맞춘 맞춤형 서비스를 제공합니다.\n- 일반 챗봇은 다양한 측면에서 개별적인 서비스를 제공합니다.\n\n### 2. **학습 및 진화**\n- AI 에이전트는 학습을 통해 인공지능으로 진화하며, 새로운 기술을 적용합니다.\n- 일반 챗봇은 기존 기능과 기존 알고리즘을 유지하며, 기능을 개선하는 과정을 거칩니다.\n\n### 3. **유저 중심**\n- AI 에이전트는 유저의 요구를 반영해 개발되고 있습니다.\n- 일반 챗봇은 기능과 기능을 유지하며, 유저의 요구를 고려하지 않습니다.\n\n### 4. **실시간 처리**\n- AI 에이전트는 실시간 처리나 빠른 반응을 가능하게 합니다.\n- 일반 챗봇은 일반적인 처리 속도와 기능을 유지하며, 실시간 처리이상의 시간이 필요할 수 있습니다.\n\n### 5. **데이터 기반**\n- AI 에이전트는 데이터를 기반으로 학습하여 개선하고 있습니다.\n- 일반 챗봇은 데이터를 기반으로 기능을 유지하며, 개선을 위한 데이터 수집이 필요합니다.\n\n### 6. **고객 피드백**\n- AI 에이전트는 고객의 피드백을 기반으로 개선하고 있습니다.\n- 일반 챗봇은 피드백을 기반으로 개선하지 않거나 개선되지 않음을 반영합니다.\n\n### 7. **기술적 전문성**\n- AI 에이전트는 기술적 전문성을 갖춘 인력으로 개발되며, 복잡한 문제를 해결할 수 있습니다.\n- 일반 챗봇은 기술적인 문제를 해결하지 못할 수 있습니다.\n\n### 8. **인공지능의 역할**\n- AI 에이전트는 인간과 협업하며, 인간의 역할을 수행합니다.\n- 일반 챗봇은 인간과 협업하며, 

In [ ]:
# ==================================================
# 실습 3: Qwen3와 Gemma3 모델 비교
# ==================================================

import requests
import time

# 리스트(list)에 두 모델 이름 저장
models = ["qwen3:0.6b", "gemma3:1b"]

# 두 모델에 동일한 질문 입력
question = "생성형 AI와 기존 인공지능의 차이를 정확히 세 문장으로 설명해줘."
URL = "http://localhost:11434/api/chat"

# for 반복문을 사용해 두 모델을 순서대로 실행
for model in models:

    payload = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ],
        "stream": False,
        "think": False,
        "options": {
            "temperature": 0,
            "num_predict": 200
        }
    }
    start = time.perf_counter()
    response = requests.post(
        URL,
        json=payload,
        timeout=180
    )
    response.raise_for_status()
    result = response.json()
    elapsed = time.perf_counter() - start

    # 모델 응답 추출
    answer = result["message"]["content"]

    print(f"모델: {model}")
    print(f"질문: {question}")
    print(f"응답:\n{answer}")
    print(f"전체 실행 시간: {elapsed:.2f}초")

모델: qwen3:0.6b
질문: 생성형 AI와 기존 인공지능의 차이를 정확히 세 문장으로 설명해줘.
응답:
생성형 AI는 기존 인공지능보다 더 구체적인 정보를 제공하는 능력을 갖는다. 예를 들어, 기존 AI는 단순한 데이터를 기반으로 모델을 생성하는 데만 도움을 줄 수 있지만, 생성형 AI는 데이터를 기반으로 모델을 생성하고, 이를 활용하여 실제 문제를 해결하는 능력을 갖는다.  

생성형 AI는 기존 인공지능보다 더 빠르게 문제를 해결할 수 있는 능력을 갖는다. 예를 들어, 기존 AI는 단순한 데이터를 기반으로 모델을 생성하는 데만 도움을 줄 수 있지만, 생성형 AI는 데이터를 기반으로 모델을 생성하고, 이를 활용하여 실제 문제를 해결하는 능력을 갖는다.
전체 실행 시간: 5.46초
모델: gemma3:1b
질문: 생성형 AI와 기존 인공지능의 차이를 정확히 세 문장으로 설명해줘.
응답:
생성형 AI와 기존 인공지능은 **목표**와 **방법**에 차이가 있습니다.** 생성형 AI는 새로운 콘텐츠를 생성하는 데 특화되어, 텍스트, 이미지, 음악 등 다양한 형태의 콘텐츠를 만들어냅니다. 기존 인공지능은 주어진 작업을 수행하거나 데이터를 분석하는 데 초점을 맞추며, 특정 목적에 맞춰 작동합니다.  결론적으로, 생성형 AI는 창의적인 결과물을 생성하는 데 강점을, 기존 인공지능은 분석 및 처리 능력을 강조합니다.
전체 실행 시간: 6.53초


In [15]:
# ==================================================
# 실습 4: Qwen3와 Gemma3 추론 성능 측정
# ==================================================

import requests
import pandas as pd

URL = "http://localhost:11434/api/chat"
models = ["qwen3:0.6b", "gemma3:1b"]
question = "파이썬에서 함수와 클래스의 차이를 설명해줘."

# 빈 리스트를 생성하여 모델별 평가 결과 저장
results = []

for model in models:

    payload = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ],
        "stream": False,
        "think": False,
        "options": {
            "temperature": 0,
            "num_predict": 200
        }
    }

    response = requests.post(
        URL,
        json=payload,
        timeout=180
    )

    response.raise_for_status()
    data = response.json()

    # 나노초(ns)를 초(s)로 변환
    total_time = data.get("total_duration", 0) / 1e9
    load_time = data.get("load_duration", 0) / 1e9

    # 출력 토큰 수와 생성 시간
    tokens = data.get("eval_count", 0)
    eval_time = data.get("eval_duration", 0) / 1e9

    # ZeroDivisionError 방지
    tokens_per_sec = (
        tokens / eval_time if eval_time > 0 else 0
    )

    # append(): 리스트에 딕셔너리 추가
    results.append({
        "모델": model,
        "전체 시간(초)": round(total_time, 2),
        "로딩 시간(초)": round(load_time, 2),
        "생성 토큰 수": tokens,
        "초당 토큰 수": round(tokens_per_sec, 2)
    })

# [Pandas]
# 리스트 안의 딕셔너리들을 표 형태로 변환

df = pd.DataFrame(results)

display(df)

,모델,전체 시간(초),로딩 시간(초),생성 토큰 수,초당 토큰 수
0,qwen3:0.6b,2.36,0.02,200,86.26
1,gemma3:1b,4.16,0.02,200,48.96


In [16]:
print(data)

{'model': 'gemma3:1b', 'created_at': '2026-11-01T11:26:51.9095064Z', 'message': {'role': 'assistant', 'content': '파이썬에서 함수와 클래스는 모두 코드의 재사용성과 구조화에 중요한 역할을 하지만, 그 역할과 특징에 차이가 있습니다. 핵심적인 차이점을 설명해 드리겠습니다.\n\n**1. 함수 (Function)**\n\n*   **정의:** 함수는 특정 작업을 수행하는 코드 블록입니다. 입력(argument)을 받아 특정 작업을 수행하고, 결과를 반환합니다.\n*   **역할:** 특정 작업을 수행하는 독립적인 코드 조각을 정의하고 실행합니다.\n*   **특징:**\n    *   **구체적인 작업 수행:** 특정 작업을 수행하는 데 집중합니다.\n    *   **입력값 처리:** 입력값을 받아 처리하고 결과를 반환합니다.\n    *   **반환값:**  함수가 반환하는 값을 가집니다.\n*   **예시:**\n    ```python\n    def greet(name):\n        print("Hello, " + name + "!")'}, 'done': True, 'done_reason': 'length', 'total_duration': 4163884100, 'load_duration': 20117100, 'prompt_eval_count': 23, 'prompt_eval_cached_count': 18, 'prompt_eval_duration': 52167000, 'eval_count': 200, 'eval_duration': 4084605000}


In [ ]:
# ==================================================
# 실습 5: LLM 평가 결과 CSV 저장
# ==================================================

# to_csv(): DataFrame을 CSV 파일로 저장
# index=False: 행 번호를 별도 열로 저장하지 않음
# utf-8-sig: Excel에서 한글 깨짐 방지

df.to_csv(
    "01_Qwen_Gemma_Comparison.csv",
    index=False,
    encoding="utf-8-sig"
)